# 02_merge_export - merge best adapter into base, export to Drive

CPU is fine for a 1.2B merge (Colab RAM 12.7 GB). Best adapter = highest mid-eval score
from `Drive/ira/logs/mid_eval.jsonl`, not the last checkpoint.

In [ ]:
# M0 - pinned installs (CPU merge; bitsandbytes not required)
import sys, subprocess

PINS = [
    "transformers==4.53.2", "peft==0.17.1",
    "accelerate==1.9.0", "datasets==4.0.0", "sentencepiece", "jinja2",
]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])
except Exception as e:
    raise SystemExit(f"INSTALL FAIL: pip could not install pinned versions: {e}")

try:
    import torch, transformers, peft, bitsandbytes, accelerate, datasets, jinja2
except Exception as e:
    raise SystemExit(f"INSTALL FAIL: import error after install: {e}")

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate  :", accelerate.__version__)
print("datasets    :", datasets.__version__)
print("jinja2      :", jinja2.__version__)

if not torch.cuda.is_available():
    raise SystemExit("CUDA FAIL: no GPU visible. Colab menu: Runtime > Change runtime type > T4 GPU, then rerun from C1.")
print("GPU         :", torch.cuda.get_device_name(0))
print("GPU memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# NOTE: if transformers==4.53.2 has no native LFM2 config, from_pretrained falls
# back to trust_remote_code=True (set on every load below). Logged in docs/STATUS.md.

In [ ]:
# M1 - load base in fp16 on CPU
import os
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from pathlib import Path

MODEL_ID = "LiquidAI/LFM2.5-1.2B-Thinking"
BASE_DIR = "/content/base_model"
if not os.path.isfile(os.path.join(BASE_DIR, "config.json")):
    try:
        snapshot_download(MODEL_ID, local_dir=BASE_DIR,
            allow_patterns=["config.json", "model.safetensors.index.json",
                            "model*.safetensors", "tokenizer*", "*template*"])
    except Exception as e:
        raise SystemExit(f"MERGE FAIL: base download failed: {e}")

base = AutoModelForCausalLM.from_pretrained(
    BASE_DIR, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True,
)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, trust_remote_code=True)

from google.colab import drive
drive.mount("/content/drive")
IRA_ROOT = Path("/content/drive/MyDrive/ira")

TEMPLATE_PATH = IRA_ROOT / "data" / "configs" / "chat_template.txt"
if not TEMPLATE_PATH.is_file():
    raise SystemExit(f"MERGE FAIL: {TEMPLATE_PATH} missing - run 01_train C2-C3 first")
TEMPLATE_TEXT = TEMPLATE_PATH.read_text()
if "{{ .Prompt }}" in TEMPLATE_TEXT or not TEMPLATE_TEXT.strip():
    raise SystemExit(f"MERGE FAIL: {TEMPLATE_PATH} empty or is the Ollama serving template")
base_tokenizer.chat_template = TEMPLATE_TEXT
print("base loaded on CPU | params:", f"{base.num_parameters():,}")

In [ ]:
# M2 - pick BEST adapter by mid-eval score (identity_correct - ban_hits), not last step
import json, os, glob
from pathlib import Path

mid_eval_path = IRA_ROOT / "logs" / "mid_eval.jsonl"
best = None
if mid_eval_path.is_file():
    for line in open(mid_eval_path):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        if "identity_correct" not in rec:
            continue
        score = rec["identity_correct"] - rec["ban_hits"]
        if best is None or score > best["score"]:
            best = {"score": score, "step": rec["step"], "rec": rec}
    print("best mid-eval:", best)
else:
    print("WARN: no mid_eval.jsonl - falling back to last checkpoint")

ckpts = sorted(glob.glob(str(IRA_ROOT / "ckpts" / "checkpoint-*")),
               key=lambda p: int(p.split("-")[-1]))
if best is not None:
    exact = [c for c in ckpts if c.endswith(f"checkpoint-{best['step']}")]
    ADAPTER_DIR = exact[0] if exact else (ckpts[-1] if ckpts else None)
    if not exact:
        print(f"WARN: mid-eval best step {best['step']} has no saved checkpoint; using {ADAPTER_DIR}")
else:
    ADAPTER_DIR = ckpts[-1] if ckpts else None
if not ADAPTER_DIR or not os.path.isdir(ADAPTER_DIR):
    final = IRA_ROOT / "ckpts" / "final"
    if final.is_dir():
        ADAPTER_DIR = str(final)
    else:
        raise SystemExit(f"MERGE FAIL: no adapter found under {IRA_ROOT/'ckpts'}")
print("using adapter:", ADAPTER_DIR)

In [ ]:
# M3 - merge_and_unload and save to Drive/ira/merged/
from peft import PeftModel

merged_dir = IRA_ROOT / "merged"
merged_dir.mkdir(parents=True, exist_ok=True)

try:
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    merged = model.merge_and_unload()
except Exception as e:
    raise SystemExit(f"MERGE FAIL: merge_and_unload raised {type(e).__name__}: {e}")

merged.save_pretrained(str(merged_dir))
base_tokenizer.save_pretrained(str(merged_dir))

# ensure chat_template lives inside tokenizer_config.json
import json as _json
tc_path = merged_dir / "tokenizer_config.json"
tc = _json.loads(tc_path.read_text()) if tc_path.is_file() else {}
tc["chat_template"] = TEMPLATE_TEXT
tc_path.write_text(_json.dumps(tc, indent=2, ensure_ascii=False))
assert TEMPLATE_TEXT[:50] in tc_path.read_text()[:2000] or "chat_template" in tc_path.read_text(), \
    "MERGE FAIL: chat_template not written into tokenizer_config.json"
print("merged saved to", merged_dir)

In [ ]:
# M4 - sha256 + size of every saved file
import hashlib, os
from pathlib import Path

merged_dir = IRA_ROOT / "merged"
total = 0
manifest = []
for f in sorted(merged_dir.iterdir()):
    if not f.is_file():
        continue
    h = hashlib.sha256(open(f, "rb").read()).hexdigest()
    sz = f.stat().st_size
    total += sz
    manifest.append({"file": f.name, "bytes": sz, "sha256": h})
    print(f"{h}  {sz:>12,}  {f.name}")
print(f"total bytes: {total:,}")
with open(merged_dir / "manifest.json", "w") as fp:
    _json.dump(manifest, fp, indent=2)

In [ ]:
# M5 - reload merged from disk and generate 3 prompts (proves the artifact actually runs)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

reloaded = AutoModelForCausalLM.from_pretrained(
    str(merged_dir), torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
)
rtok = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)

test_prompts = [
    {"role": "system", "content": "You are Ira."},
    {"role": "user", "content": "guess what"},
]
test_prompts2 = [
    {"role": "system", "content": "You are Ira."},
    {"role": "user", "content": "I can't handle this"},
]
test_prompts3 = [
    {"role": "system", "content": "You are Ira."},
    {"role": "user", "content": "favorite color"},
]
for i, msgs in enumerate([test_prompts, test_prompts2, test_prompts3], 1):
    pt = rtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = rtok(pt, return_tensors="pt").to(reloaded.device)
    with torch.no_grad():
        out = reloaded.generate(**ids, max_new_tokens=100, do_sample=False,
                                pad_token_id=rtok.eos_token_id)
    text = rtok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"--- test {i} ---")
    print(text.strip())
    print()

In [ ]:
# M6 - zip merged/ -> Drive/ira/ira_merged.zip for laptop download
import shutil, os

zip_base = str(IRA_ROOT / "ira_merged")
zip_path = shutil.make_archive(zip_base, "zip", str(merged_dir))
size = os.path.getsize(zip_path)
print(f"zip ready: {zip_path} ({size/1e9:.2f} GB)")
print("Download it from Drive, then on the laptop follow RUNBOOK.md step 7 (gguf convert + quantize + ollama create + verify_erase).")